# Minimal working example using BERT (`distBERT` model)

## Setup

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

c:\Users\flowe\miniconda3\envs\de300\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pseudo data creation

In [14]:
# ----------------------------
# 1) Read data from folder
# ----------------------------

data_dir = '../datasets'
items = pd.read_csv(data_dir + '/items.csv')
events = pd.read_csv(data_dir + '/events.csv')

## BERT encoder (What makes BERT work?)

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3919.18it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [5]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=128):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [15]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = (
    f'Title: {items['title']}. Description: {items['description']}. Category: {items['category']}. Brand: {items['brand']}. Tags: {items['tags']}. Price: {items['price']}.'
    )
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


## What user-specific data are there?

In [26]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=3, min_dwell=10, allowed_event=None):
    user_events = events[events["user_id"] == user_id].copy()
    if allowed_event is not None:
        user_events = user_events[user_events['event_type'].isin(allowed_event)]
    if min_dwell is not None:
        user_events = user_events[user_events['dwell_seconds'] >= min_dwell]
        
    hist = (user_events[user_events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history"
    text = items.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)


## How to recommend "similar" item with BERT?

In [24]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    user_text, seen = build_user_text(user_id, events, items, N=3, allowed_event=['view', 'add_to_cart', 'purchase'])
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs


In [27]:

for i in range(1, 11):
    print(f'u{i}', "->", recommend(f'u{i}', k=5))


u1 -> ['i7', 'i6', 'i4', 'i2', 'i1']
u2 -> ['i8', 'i7', 'i6', 'i3', 'i2']
u3 -> ['i7', 'i6', 'i5', 'i4', 'i3']
u4 -> ['i8', 'i7', 'i6', 'i5', 'i4']
u5 -> ['i8', 'i7', 'i6', 'i4', 'i3']
u6 -> ['i8', 'i7', 'i6', 'i5', 'i3']
u7 -> ['i8', 'i7', 'i6', 'i5', 'i4']
u8 -> ['i8', 'i7', 'i6', 'i5', 'i4']
u9 -> ['i7', 'i6', 'i5', 'i4', 'i3']
u10 -> ['i8', 'i7', 'i5', 'i4', 'i3']
